# Fase 3 — Notebook 2: Mini CV para Selección de Modelo

**Objetivo:** Comparar 3 configuraciones candidatas usando solo el conjunto
de train real (sin sintéticos) para elegir el modelo que se entrenará
con augmentation en el notebook 4.

**Candidatos:**
| # | Modelo | Filtro | Duración |
|---|--------|--------|----------|
| 1 | CNN-2D | log-mel | 2s |
| 2 | AST-B  | gammatone | 2s |
| 3 | AST-B  | gammatone | 4s |

**CV:** RepeatedStratifiedKFold — 3 folds × 2 repeticiones = **6 folds**  
**Datos:** Solo stems de train del split holdout (test nunca se toca)  
**Métrica de decisión:** F1-Macro (media sobre 6 folds)  
**Threshold:** dinámico por fold (barrido F1-macro sobre val interna 15%)  

**Prerequisito:** correr `fase3_split_holdout.ipynb` primero.

In [1]:
# ─── Celda 1: Imports y configuración ─────────────────────────────────────
import gc
import re
import json
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import ASTConfig, ASTModel

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc
)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════════════
CONDITION    = "Ansiedad"
SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
SPLITS_DIR   = Path("Ansiedad/fase3/splits")
RESULTS_ROOT = Path("Ansiedad/fase3/mini_cv")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Mini CV: 3 folds x 2 reps = 6 folds (suficiente para seleccion, rapido)
N_SPLITS  = 3
N_REPEATS = 2
VAL_FRAC  = 0.15
THR_GRID  = np.arange(0.05, 0.96, 0.02)

# Hiperparametros CNN-2D
CNN_BATCH  = 128
CNN_EPOCHS = 30
CNN_PATIENCE = 5

# Hiperparametros AST
AST_BATCH    = 32
AST_EPOCHS   = 20
AST_PATIENCE = 4
AST_LR       = 2e-5
AST_N_UNFREEZE = 3
AST_PRETRAINED = "MIT/ast-finetuned-audioset-10-10-0.4593"
OLD_FREQ_PATCHES = 12
OLD_TIME_PATCHES = 101

# Los 3 candidatos
CANDIDATOS = [
    {"id": "CNN2D_logmel_2s",     "model": "CNN2D", "filter": "log-mel",   "dur": 2, "img_h": 128},
    {"id": "AST_gammatone_2s",    "model": "AST",   "filter": "gammatone", "dur": 2, "img_h": 128, "ast_h": 128},
    {"id": "AST_gammatone_3s",    "model": "AST",   "filter": "gammatone", "dur": 3, "img_h": 128, "ast_h": 128},
    {"id": "AST_gammatone_4s",    "model": "AST",   "filter": "gammatone", "dur": 4, "img_h": 128, "ast_h": 128},
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"\nCV: {N_SPLITS} folds x {N_REPEATS} reps = {N_SPLITS*N_REPEATS} folds por candidato")
print(f"Candidatos: {[c['id'] for c in CANDIDATOS]}")

/home/ci2dt2-ai/Proyectos/Psiquiatria/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080

CV: 3 folds x 2 reps = 6 folds por candidato
Candidatos: ['CNN2D_logmel_2s', 'AST_gammatone_2s', 'AST_gammatone_4s']


In [2]:
# ─── Celda 2: Utilidades de datos ─────────────────────────────────────────

def load_split(filter_name, duration):
    """Carga el JSON de split generado por el notebook 1."""
    json_path = SPLITS_DIR / f"split_{filter_name}_{duration}s.json"
    if not json_path.exists():
        raise FileNotFoundError(
            f"Split no encontrado: {json_path}\n"
            "Corre primero fase3_split_holdout.ipynb")
    with open(json_path) as f:
        sp = json.load(f)
    train_stems  = sp["train"]["stems"]
    train_labels = np.array(sp["train"]["labels"], dtype=np.int64)
    print(f"  Split cargado: {filter_name} {duration}s — "
          f"train={len(train_stems)} pac "
          f"(0:{sp['train']['n_class0']} / 1:{sp['train']['n_class1']})")
    return train_stems, train_labels


def build_img_map(root: Path):
    """Construye mapa stem -> [paths de segmentos]."""
    img_map = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
    return img_map


# ── Transforms CNN-2D (sin resize) ────────────────────────────────────────
CNN_TF_TRAIN = T.Compose([
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
CNN_TF_VAL = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Transform AST (escala de grises) ──────────────────────────────────────
AST_TF = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])


# ── Datasets ───────────────────────────────────────────────────────────────
class CNNDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = []
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), \
               torch.tensor(label, dtype=torch.long)


class CNNStemDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = [(p, s) for s in stems for p in img_map[s]]
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), stem


class ASTDataset(Dataset):
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = []
        self.ast_h    = ast_h
        self.target_w = target_w
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        t = AST_TF(Image.open(path)).squeeze(0)
        if t.shape[0] < self.ast_h:
            t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
        if t.shape[1] < self.target_w:
            t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
        return t.T, torch.tensor(label, dtype=torch.long)


class ASTStemDataset(Dataset):
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = [(p, s) for s in stems for p in img_map[s]]
        self.ast_h    = ast_h
        self.target_w = target_w

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        t = AST_TF(Image.open(path)).squeeze(0)
        if t.shape[0] < self.ast_h:
            t = F.pad(t, (0, 0, 0, self.ast_h - t.shape[0]), value=-1.0)
        if t.shape[1] < self.target_w:
            t = F.pad(t, (0, self.target_w - t.shape[1]), value=-1.0)
        return t.T, stem


def get_target_width(img_map, stems):
    valid = [img_map[s][0] for s in stems if s in img_map]
    return max(Image.open(p).size[0] for p in valid) if valid else 63


print("Datasets y transforms definidos")

Datasets y transforms definidos


In [3]:
# ─── Celda 3: Modelos ──────────────────────────────────────────────────────

class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()
        def conv_block(ic, oc, n):
            lys = []
            for i in range(n):
                lys += [nn.Conv2d(ic if i==0 else oc, oc, 3, padding=1),
                        nn.BatchNorm2d(oc), nn.ReLU(inplace=True)]
            return nn.Sequential(*lys, nn.MaxPool2d(2), nn.Dropout2d(0.2))
        self.features = nn.Sequential(
            conv_block(3,   32,  2), conv_block(32,  64,  2),
            conv_block(64,  128, 3), conv_block(128, 256, 3),
            conv_block(256, 256, 3))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128,  64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64,    1), nn.Sigmoid())
    def forward(self, x):
        return self.classifier(self.pool(self.features(x))).squeeze(1)


class ASTClassifier(nn.Module):
    def __init__(self, ast_h, target_w, n_unfreeze=AST_N_UNFREEZE):
        super().__init__()
        fp = (ast_h    - 16) // 10 + 1
        tp = (target_w - 16) // 10 + 1
        print(f"    AST patches: freq={fp} time={tp} total={fp*tp}")
        config = ASTConfig(
            num_mel_bins=ast_h, max_length=target_w,
            hidden_size=768, num_hidden_layers=12,
            num_attention_heads=12, intermediate_size=3072,
            patch_size=16, frequency_stride=10, time_stride=10)
        pretrained = ASTModel.from_pretrained(AST_PRETRAINED)
        self.ast   = ASTModel(config)
        self.ast.encoder.load_state_dict(pretrained.encoder.state_dict())
        self.ast.layernorm.load_state_dict(pretrained.layernorm.state_dict())
        self.ast.embeddings.patch_embeddings.load_state_dict(
            pretrained.embeddings.patch_embeddings.state_dict())
        self.ast.embeddings.cls_token.data = pretrained.embeddings.cls_token.data.clone()
        self.ast.embeddings.distillation_token.data = \
            pretrained.embeddings.distillation_token.data.clone()
        old_pos = pretrained.embeddings.position_embeddings.data
        self.ast.embeddings.position_embeddings = nn.Parameter(
            self._interp(old_pos, fp, tp, OLD_FREQ_PATCHES, OLD_TIME_PATCHES))
        del pretrained; gc.collect()
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, 1),   nn.Sigmoid())
        for p in self.parameters(): p.requires_grad = False
        for layer in self.ast.encoder.layer[-n_unfreeze:]:
            for p in layer.parameters(): p.requires_grad = True
        for p in self.ast.layernorm.parameters(): p.requires_grad = True
        for p in self.classifier.parameters():   p.requires_grad = True
        n_tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Entrenables: {n_tr:,}")

    @staticmethod
    def _interp(old, nf, nt, of, ot):
        sp = old[:, :2, :]; pt = old[:, 2:, :]; h = pt.shape[-1]
        pt = pt.reshape(1,of,ot,h).permute(0,3,1,2)
        pt = F.interpolate(pt.float(), size=(nf,nt), mode='bilinear', align_corners=False)
        return torch.cat([sp, pt.permute(0,2,3,1).reshape(1,nf*nt,h)], dim=1)

    def forward(self, x):
        return self.classifier(
            self.ast(input_values=x).last_hidden_state[:, 0, :]).squeeze(1)


print("Modelos CNN2D y ASTClassifier definidos")

Modelos CNN2D y ASTClassifier definidos


In [4]:
# ─── Celda 4: Funciones de entrenamiento e inferencia ─────────────────────

def train_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        loss = (criterion(model(xb).view(-1), yb) * cw_tensor[yb.long()]).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def predict_stems(model, stems, img_map, dataset_cls, batch_size, **ds_kwargs):
    model.eval()
    ds = dataset_cls(stems, img_map, **ds_kwargs)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)
    probs = {s: [] for s in stems}
    with torch.no_grad():
        for xb, snames in dl:
            out = model(xb.to(DEVICE, non_blocking=True)).view(-1).cpu().numpy()
            for p, s in zip(out, snames): probs[s].append(float(p))
    return {s: float(np.mean(v)) for s, v in probs.items()}


def calibrate_thr(probs_dict, stems, labels):
    arr = np.array([probs_dict[s] for s in stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        sc = f1_score(labels, (arr > thr).astype(int), average='macro', zero_division=0)
        if sc > best_f1: best_f1, best_thr = sc, float(thr)
    return best_thr


def eval_fold(probs_dict, stems, labels, thr):
    true = [int(l) for l in labels]
    pred = [1 if probs_dict[s] > thr else 0 for s in stems]
    prob = [probs_dict[s] for s in stems]
    true_arr, prob_arr = np.array(true), np.array(prob)
    if len(np.unique(true_arr)) > 1:
        fpr, tpr, _ = roc_curve(true_arr, prob_arr)
        fold_auc = float(auc(fpr, tpr))
    else:
        fold_auc = 0.0
    return {
        "acc"   : float(accuracy_score(true, pred)),
        "prec"  : float(precision_score(true, pred, zero_division=0)),
        "rec"   : float(recall_score(true, pred, zero_division=0)),
        "f1"    : float(f1_score(true, pred, zero_division=0)),
        "f1mac" : float(f1_score(true, pred, average='macro', zero_division=0)),
        "auc"   : fold_auc,
        "threshold": round(thr, 4),
    }


print("Funciones de entrenamiento e inferencia definidas")

Funciones de entrenamiento e inferencia definidas


In [ ]:
# ─── Celda 5: LOOP PRINCIPAL — mini CV sobre los 4 candidatos ─────────────

resultados = {}   # { candidato_id: [fold_metrics...] }
ckpt_global = RESULTS_ROOT / "mini_cv_checkpoint.json"

# Cargar progreso previo si existe
if ckpt_global.exists():
    with open(ckpt_global) as f:
        resultados = json.load(f)
    print(f"[CKPT] Progreso cargado: {list(resultados.keys())}")

for cand in CANDIDATOS:
    cid      = cand["id"]
    model_t  = cand["model"]
    filt     = cand["filter"]
    dur      = cand["dur"]
    img_h    = cand["img_h"]

    if cid in resultados and len(resultados[cid]) >= N_SPLITS * N_REPEATS:
        print(f"\n[SKIP] {cid} ya completado ({len(resultados[cid])} folds)")
        continue

    folds_done = len(resultados.get(cid, []))

    print(f"\n{'='*72}")
    print(f"  CANDIDATO: {cid}")
    print(f"  Modelo={model_t}  Filtro={filt}  Dur={dur}s")
    print(f"{'='*72}")

    # Cargar split y construir img_map
    train_stems, train_labels = load_split(filt, dur)
    image_root = SPECS_ROOT / filt / f"{dur}s"
    img_map    = build_img_map(image_root)

    total_segs = sum(len(img_map[s]) for s in train_stems if s in img_map)
    print(f"  Segmentos train totales: {total_segs:,}")

    if model_t == "AST":
        target_w = get_target_width(
            {s: img_map[s] for s in train_stems if s in img_map})
        ast_h    = cand["ast_h"]
        print(f"  AST input: ({target_w}, {ast_h})")

    # Mini CV
    rskf = RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)
    stems_arr = np.array(train_stems)
    fold_metrics_list = list(resultados.get(cid, []))

    for fold_idx, (tr_idx, te_idx) in enumerate(
            rskf.split(stems_arr, train_labels)):

        if fold_idx < folds_done:
            continue

        rep_n    = fold_idx // N_SPLITS + 1
        fold_n   = fold_idx %  N_SPLITS + 1
        fold_g   = fold_idx + 1
        total_f  = N_SPLITS * N_REPEATS
        print(f"\n  Rep {rep_n}/{N_REPEATS} | Fold {fold_n}/{N_SPLITS} [{fold_g}/{total_f}]")

        tr_stems  = list(stems_arr[tr_idx])
        te_stems  = list(stems_arr[te_idx])
        tr_labels = train_labels[tr_idx]
        te_labels = train_labels[te_idx]

        # Val interna 15% para calibrar threshold
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=VAL_FRAC, random_state=42+fold_idx)
        tr_sub_idx, val_idx = next(sss.split(tr_stems, tr_labels))
        val_stems       = [tr_stems[i] for i in val_idx]
        val_labels_fold = tr_labels[val_idx]
        train_sub       = [tr_stems[i] for i in tr_sub_idx]
        train_sub_lbl   = tr_labels[tr_sub_idx]

        # FIX: target_w calculado SOLO con stems de este fold
        fold_img_map = {s: img_map[s] for s in train_sub if s in img_map}
        tw_fold = get_target_width(fold_img_map, train_sub) if model_t=="AST" else None

        # ── Construir modelo, optimizer y dataloader según tipo ──────────
        if model_t == "CNN2D":
            batch_size = CNN_BATCH
            epochs     = CNN_EPOCHS
            patience   = CNN_PATIENCE
            train_ds   = CNNDataset(train_sub, fold_img_map, CNN_TF_TRAIN)
            train_dl   = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                    num_workers=4, pin_memory=True,
                                    persistent_workers=False)
            model      = CNN2D().to(DEVICE)
            optimizer  = optim.Adam(model.parameters(), lr=1e-4)
            scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
                             optimizer, 'min', factor=0.5, patience=3, min_lr=1e-7)
            infer_kwargs_val  = {"dataset_cls": CNNStemDataset, "batch_size": batch_size,
                                 "transform": CNN_TF_VAL}
            infer_kwargs_test = infer_kwargs_val

        else:  # AST
            batch_size = AST_BATCH
            epochs     = AST_EPOCHS
            patience   = AST_PATIENCE
            train_ds   = ASTDataset(train_sub, img_map, ast_h, target_w)
            train_dl   = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                    num_workers=4, pin_memory=True,
                                    persistent_workers=False)
            model      = ASTClassifier(ast_h, target_w).to(DEVICE)
            optimizer  = optim.AdamW(model.parameters(), lr=AST_LR, weight_decay=0.01)
            scheduler  = optim.lr_scheduler.CosineAnnealingLR(
                             optimizer, T_max=epochs, eta_min=1e-7)
            infer_kwargs_val  = {"dataset_cls": ASTStemDataset, "batch_size": batch_size,
                                 "ast_h": ast_h, "target_w": target_w}
            infer_kwargs_test = infer_kwargs_val

        weights   = class_weight.compute_class_weight(
                        'balanced', classes=np.unique(train_sub_lbl), y=train_sub_lbl)
        cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
        criterion = nn.BCELoss(reduction='none')

        print(f"  Train: {len(train_sub)} pac / {len(train_ds):,} segs  "
              f"| Val: {len(val_stems)} pac  | Test: {len(te_stems)} pac")

        # ── Entrenamiento ────────────────────────────────────────────────
        best_loss, pat_cnt, best_state = float('inf'), 0, None
        t0 = time.time()

        for epoch in range(epochs):
            loss = train_epoch(model, train_dl, optimizer, criterion, cw_tensor)
            if model_t == "CNN2D":
                scheduler.step(loss)
            else:
                scheduler.step()
            if loss < best_loss:
                best_loss, pat_cnt = loss, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                mark = "v"
            else:
                pat_cnt += 1
                mark = f"({pat_cnt}/{patience})"
            print(f"  Ep {epoch+1:02d}/{epochs}  loss {loss:.4f}  "
                  f"best {best_loss:.4f}  {mark}")
            if pat_cnt >= patience:
                print("  Early stopping.")
                break

        model.load_state_dict(best_state)
        print(f"  Entrenamiento: {time.time()-t0:.1f}s")

        # ── Calibrar threshold y evaluar ─────────────────────────────────
        val_probs  = predict_stems(model, val_stems, img_map, **infer_kwargs_val)
        best_thr   = calibrate_thr(val_probs, val_stems, val_labels_fold)
        test_probs = predict_stems(model, te_stems, img_map, **infer_kwargs_test)
        metrics    = eval_fold(test_probs, te_stems, te_labels, best_thr)

        print(f"  thr={best_thr:.2f}  acc={metrics['acc']:.3f}  "
              f"prec={metrics['prec']:.3f}  rec={metrics['rec']:.3f}  "
              f"F1={metrics['f1']:.3f}  F1-mac={metrics['f1mac']:.3f}  "
              f"AUC={metrics['auc']:.3f}")

        fold_metrics_list.append(metrics)
        resultados[cid] = fold_metrics_list
        with open(ckpt_global, "w") as f:
            json.dump(resultados, f)

        del model, train_ds, train_dl, best_state, cw_tensor
        gc.collect()
        torch.cuda.empty_cache()

print("\nMini CV completada")


  CANDIDATO: CNN2D_logmel_2s
  Modelo=CNN2D  Filtro=log-mel  Dur=2s
  Split cargado: log-mel 2s — train=63 pac (0:48 / 1:15)
  Segmentos train totales: 10,448

  Rep 1/2 | Fold 1/3 [1/6]
  Train: 35 pac / 5,845 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/30  loss 0.7684  best 0.7684  v
  Ep 02/30  loss 0.7549  best 0.7549  v
  Ep 03/30  loss 0.7530  best 0.7530  v
  Ep 04/30  loss 0.7480  best 0.7480  v
  Ep 05/30  loss 0.7415  best 0.7415  v
  Ep 06/30  loss 0.7338  best 0.7338  v
  Ep 07/30  loss 0.7115  best 0.7115  v
  Ep 08/30  loss 0.6785  best 0.6785  v
  Ep 09/30  loss 0.6390  best 0.6390  v
  Ep 10/30  loss 0.6007  best 0.6007  v
  Ep 11/30  loss 0.5550  best 0.5550  v
  Ep 12/30  loss 0.5348  best 0.5348  v
  Ep 13/30  loss 0.5117  best 0.5117  v
  Ep 14/30  loss 0.4773  best 0.4773  v
  Ep 15/30  loss 0.4542  best 0.4542  v
  Ep 16/30  loss 0.4306  best 0.4306  v
  Ep 17/30  loss 0.4004  best 0.4004  v
  Ep 18/30  loss 0.3862  best 0.3862  v
  Ep 19/30  loss 0.3802  best 0.3

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6732.16it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Entrenables: 21,462,273
  Train: 35 pac / 5,845 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/20  loss 0.6958  best 0.6958  v
  Ep 02/20  loss 0.5961  best 0.5961  v
  Ep 03/20  loss 0.4924  best 0.4924  v
  Ep 04/20  loss 0.4111  best 0.4111  v
  Ep 05/20  loss 0.3396  best 0.3396  v
  Ep 06/20  loss 0.2757  best 0.2757  v
  Ep 07/20  loss 0.1932  best 0.1932  v
  Ep 08/20  loss 0.1593  best 0.1593  v
  Ep 09/20  loss 0.1077  best 0.1077  v
  Ep 10/20  loss 0.0692  best 0.0692  v
  Ep 11/20  loss 0.0418  best 0.0418  v
  Ep 12/20  loss 0.0165  best 0.0165  v
  Ep 13/20  loss 0.0110  best 0.0110  v
  Ep 14/20  loss 0.0041  best 0.0041  v
  Ep 15/20  loss 0.0009  best 0.0009  v
  Ep 16/20  loss 0.0003  best 0.0003  v
  Ep 17/20  loss 0.0002  best 0.0002  v
  Ep 18/20  loss 0.0002  best 0.0002  v
  Ep 19/20  loss 0.0002  best 0.0002  v
  Ep 20/20  loss 0.0002  best 0.0002  v
  Entrenamiento: 69.6s
  thr=0.27  acc=0.667  prec=0.333  rec=0.400  F1=0.364  F1-mac=0.569  AUC=0.588

  Rep 1/2

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20799.07it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Entrenables: 21,462,273
  Train: 35 pac / 5,898 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/20  loss 0.7090  best 0.7090  v
  Ep 02/20  loss 0.6448  best 0.6448  v
  Ep 03/20  loss 0.5557  best 0.5557  v
  Ep 04/20  loss 0.4423  best 0.4423  v
  Ep 05/20  loss 0.3295  best 0.3295  v
  Ep 06/20  loss 0.2375  best 0.2375  v
  Ep 07/20  loss 0.1729  best 0.1729  v
  Ep 08/20  loss 0.1102  best 0.1102  v
  Ep 09/20  loss 0.0653  best 0.0653  v
  Ep 10/20  loss 0.0333  best 0.0333  v
  Ep 11/20  loss 0.0231  best 0.0231  v
  Ep 12/20  loss 0.0101  best 0.0101  v
  Ep 13/20  loss 0.0033  best 0.0033  v
  Ep 14/20  loss 0.0013  best 0.0013  v
  Ep 15/20  loss 0.0002  best 0.0002  v
  Ep 16/20  loss 0.0001  best 0.0001  v
  Ep 17/20  loss 0.0001  best 0.0001  v
  Ep 18/20  loss 0.0001  best 0.0001  v
  Ep 19/20  loss 0.0001  best 0.0001  v
  Ep 20/20  loss 0.0001  best 0.0001  (1/4)
  Entrenamiento: 69.0s
  thr=0.17  acc=0.524  prec=0.308  rec=0.800  F1=0.444  F1-mac=0.514  AUC=0.700

  Rep

...

    Entrenables: 21,462,273
  Train: 35 pac / 2,753 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/20  loss 0.6283  best 0.6283  v
  Ep 02/20  loss 0.4927  best 0.4927  v
  Ep 03/20  loss 0.4077  best 0.4077  v
  Ep 04/20  loss 0.3531  best 0.3531  v
  Ep 05/20  loss 0.2669  best 0.2669  v
  Ep 06/20  loss 0.2231  best 0.2231  v
  Ep 07/20  loss 0.1726  best 0.1726  v
  Ep 08/20  loss 0.1416  best 0.1416  v
  Ep 09/20  loss 0.0904  best 0.0904  v
  Ep 10/20  loss 0.0347  best 0.0347  v
  Ep 11/20  loss 0.0145  best 0.0145  v
  Ep 12/20  loss 0.0080  best 0.0080  v
  Ep 13/20  loss 0.0039  best 0.0039  v
  Ep 14/20  loss 0.0020  best 0.0020  v
  Ep 15/20  loss 0.0017  best 0.0017  v
  Ep 16/20  loss 0.0020  best 0.0017  (1/4)
  Ep 17/20  loss 0.0020  best 0.0017  (2/4)
  Ep 18/20  loss 0.0017  best 0.0017  (3/4)
  Ep 19/20  loss 0.0015  best 0.0015  v
  Ep 20/20  loss 0.0017  best 0.0015  (1/4)
  Entrenamiento: 68.4s
  thr=0.37  acc=0.714  prec=0.333  rec=0.200  F1=0.250  F1-mac=0.537  AUC=

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18145.71it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Entrenables: 21,462,273
  Train: 35 pac / 3,045 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/20  loss 0.6796  best 0.6796  v
  Ep 02/20  loss 0.5586  best 0.5586  v
  Ep 03/20  loss 0.3971  best 0.3971  v
  Ep 04/20  loss 0.2955  best 0.2955  v
  Ep 05/20  loss 0.2053  best 0.2053  v
  Ep 06/20  loss 0.1382  best 0.1382  v
  Ep 07/20  loss 0.0947  best 0.0947  v
  Ep 08/20  loss 0.0437  best 0.0437  v
  Ep 09/20  loss 0.0178  best 0.0178  v
  Ep 10/20  loss 0.0072  best 0.0072  v
  Ep 11/20  loss 0.0024  best 0.0024  v
  Ep 12/20  loss 0.0012  best 0.0012  v
  Ep 13/20  loss 0.0004  best 0.0004  v
  Ep 14/20  loss 0.0003  best 0.0003  v
  Ep 15/20  loss 0.0003  best 0.0003  (1/4)
  Ep 16/20  loss 0.0003  best 0.0003  v
  Ep 17/20  loss 0.0003  best 0.0003  v
  Ep 18/20  loss 0.0003  best 0.0003  (1/4)
  Ep 19/20  loss 0.0002  best 0.0002  v
  Ep 20/20  loss 0.0003  best 0.0002  (1/4)
  Entrenamiento: 75.3s
  thr=0.09  acc=0.571  prec=0.333  rec=0.800  F1=0.471  F1-mac=0.555  AUC=0.57

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14482.94it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Entrenables: 21,462,273
  Train: 35 pac / 2,839 segs  | Val: 7 pac  | Test: 21 pac
  Ep 01/20  loss 0.7544  best 0.7544  v
  Ep 02/20  loss 0.6677  best 0.6677  v
  Ep 03/20  loss 0.5731  best 0.5731  v
  Ep 04/20  loss 0.4375  best 0.4375  v
  Ep 05/20  loss 0.3394  best 0.3394  v
  Ep 06/20  loss 0.2268  best 0.2268  v
  Ep 07/20  loss 0.1552  best 0.1552  v
  Ep 08/20  loss 0.0923  best 0.0923  v
  Ep 09/20  loss 0.0713  best 0.0713  v
  Ep 10/20  loss 0.0386  best 0.0386  v
  Ep 11/20  loss 0.0153  best 0.0153  v
  Ep 12/20  loss 0.0048  best 0.0048  v
  Ep 13/20  loss 0.0011  best 0.0011  v
  Ep 14/20  loss 0.0005  best 0.0005  v
  Ep 15/20  loss 0.0004  best 0.0004  v
  Ep 16/20  loss 0.0004  best 0.0004  v
  Ep 17/20  loss 0.0003  best 0.0003  v
  Ep 18/20  loss 0.0003  best 0.0003  v
  Ep 19/20  loss 0.0003  best 0.0003  v
  Ep 20/20  loss 0.0003  best 0.0003  v
  Entrenamiento: 70.6s
  thr=0.15  acc=0.429  prec=0.267  rec=0.800  F1=0.400  F1-mac=0.427  AUC=0.612

Mini CV c

In [6]:
# ─── Celda 6: Tabla comparativa y selección del modelo ganador ────────────

print("="*72)
print("RESULTADOS MINI CV — SELECCION DE MODELO")
print(f"{N_SPLITS}-Fold x {N_REPEATS} reps = {N_SPLITS*N_REPEATS} folds por candidato")
print("="*72)

keys = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
labels_m = ["Accuracy", "Precision", "Recall", "F1 (pos)", "F1-Macro", "AUC-ROC"]

resumen = {}
for cid, folds in resultados.items():
    arr = {k: np.array([m[k] for m in folds]) for k in keys}
    resumen[cid] = {k: (arr[k].mean(), arr[k].std()) for k in keys}

# Tabla por metrica
print(f"\n  {'Candidato':25s}  {'F1-Macro':18s}  {'AUC-ROC':18s}  "
      f"{'Accuracy':18s}  {'Recall':18s}")
print(f"  {'─'*85}")
for cid in resumen:
    r = resumen[cid]
    print(f"  {cid:25s}  "
          f"{r['f1mac'][0]:.3f} +/- {r['f1mac'][1]:.3f}   "
          f"{r['auc'][0]:.3f} +/- {r['auc'][1]:.3f}   "
          f"{r['acc'][0]:.3f} +/- {r['acc'][1]:.3f}   "
          f"{r['rec'][0]:.3f} +/- {r['rec'][1]:.3f}")

# Detalle por fold
print("\n  DETALLE POR FOLD:")
for cid, folds in resultados.items():
    print(f"\n  {cid}")
    print(f"    {'Fold':6s}  {'Thr':5s}  Acc    Prec   Rec    F1     F1-Mac AUC")
    for i, m in enumerate(folds):
        rep_n  = i // N_SPLITS + 1
        fold_n = i %  N_SPLITS + 1
        print(f"    R{rep_n:02d}F{fold_n:02d}  {m['threshold']:.2f}   "
              f"{m['acc']:.3f}  {m['prec']:.3f}  {m['rec']:.3f}  "
              f"{m['f1']:.3f}  {m['f1mac']:.3f}  {m['auc']:.3f}")

# Seleccion del ganador (mayor F1-Macro medio)
ganador = max(resumen, key=lambda c: resumen[c]["f1mac"][0])
ganador_info = next(c for c in CANDIDATOS if c["id"] == ganador)

print(f"\n{'='*72}")
print(f"  MODELO GANADOR: {ganador}")
print(f"  F1-Macro: {resumen[ganador]['f1mac'][0]:.3f} +/- {resumen[ganador]['f1mac'][1]:.3f}")
print(f"  AUC-ROC : {resumen[ganador]['auc'][0]:.3f} +/- {resumen[ganador]['auc'][1]:.3f}")
print(f"")
print(f"  Parametros para los notebooks siguientes:")
print(f"    model_type  = '{ganador_info['model']}'")
print(f"    filter_name = '{ganador_info['filter']}'")
print(f"    duration    = {ganador_info['dur']}")
print(f"{'='*72}")

# Guardar ganador en JSON para que notebooks 3 y 4 lo lean automaticamente
winner_path = RESULTS_ROOT / "modelo_ganador.json"
with open(winner_path, "w") as f:
    json.dump({
        "id"         : ganador,
        "model_type" : ganador_info["model"],
        "filter_name": ganador_info["filter"],
        "duration"   : ganador_info["dur"],
        "img_h"      : ganador_info["img_h"],
        "ast_h"      : ganador_info.get("ast_h"),
        "f1mac_mean" : resumen[ganador]["f1mac"][0],
        "f1mac_std"  : resumen[ganador]["f1mac"][1],
        "auc_mean"   : resumen[ganador]["auc"][0],
        "auc_std"    : resumen[ganador]["auc"][1],
    }, f, indent=2)
print(f"  Ganador guardado en: {winner_path}")

RESULTADOS MINI CV — SELECCION DE MODELO
3-Fold x 2 reps = 6 folds por candidato

  Candidato                  F1-Macro            AUC-ROC             Accuracy            Recall            
  ─────────────────────────────────────────────────────────────────────────────────────
  CNN2D_logmel_2s            0.432 +/- 0.089   0.458 +/- 0.090   0.508 +/- 0.162   0.533 +/- 0.320
  AST_gammatone_2s           0.518 +/- 0.071   0.592 +/- 0.051   0.587 +/- 0.131   0.533 +/- 0.221
  AST_gammatone_4s           0.529 +/- 0.067   0.596 +/- 0.055   0.611 +/- 0.101   0.500 +/- 0.252

  DETALLE POR FOLD:

  CNN2D_logmel_2s
    Fold    Thr    Acc    Prec   Rec    F1     F1-Mac AUC
    R01F01  0.39   0.667  0.333  0.400  0.364  0.569  0.475
    R01F02  0.13   0.381  0.250  0.800  0.381  0.381  0.412
    R01F03  0.17   0.476  0.312  1.000  0.476  0.476  0.450
    R02F01  0.59   0.762  0.000  0.000  0.000  0.432  0.613
    R02F02  0.17   0.476  0.250  0.600  0.353  0.456  0.487
    R02F03  0.07   0.286  0

In [7]:
# ─── Celda 7: Figura comparativa ──────────────────────────────────────────

keys_plot   = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
labels_plot = ["Accuracy", "Precision", "Recall", "F1 (pos)", "F1-Macro", "AUC-ROC"]
colores     = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 6, figsize=(22, 5))
fig.suptitle("Mini CV — Comparación de candidatos (Fase 3)", fontsize=13, y=1.01)

cids = list(resultados.keys())
for ax, k, lbl in zip(axes, keys_plot, labels_plot):
    data  = [np.array([m[k] for m in resultados[c]]) for c in cids]
    short = [c.replace("CNN2D_","CNN ").replace("AST_","AST ") for c in cids]
    bp = ax.boxplot(data, patch_artist=True, labels=short,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(lbl, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', labelsize=7)
    # Marcar el ganador
    if ganador in cids:
        win_idx = cids.index(ganador) + 1
        ax.axvline(win_idx, color='red', linewidth=0.8, linestyle='--', alpha=0.5)

plt.tight_layout()
fig_path = RESULTS_ROOT / "comparacion_candidatos.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f"Figura guardada: {fig_path}")
print(f"\nGANADOR: {ganador}")
print("Continuar con: fase3_vae_train_generate.ipynb")

/tmp/ipykernel_74282/2144429650.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=short,
/tmp/ipykernel_74282/2144429650.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=short,
/tmp/ipykernel_74282/2144429650.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, patch_artist=True, labels=short,
/tmp/ipykernel_74282/2144429650.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.1

Figura guardada: Ansiedad/fase3/mini_cv/comparacion_candidatos.png

GANADOR: AST_gammatone_4s
Continuar con: fase3_vae_train_generate.ipynb


/tmp/ipykernel_74282/2144429650.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
